# Download images with padding

In [15]:
import pandas as pd
from PIL import Image
import requests
from io import BytesIO

In [16]:
artworks = pd.read_csv("./Donnees/Artworks.csv")
artworks = artworks[~artworks["OnView"].isna()]
artworks = artworks[~artworks.ImageURL.isna()].reset_index(drop = True)

In [17]:
len(artworks.ImageURL)

1147

In [18]:
for i, row in artworks.iterrows():
    image_url = row["ImageURL"]
    headers = {
                "User-Agent": "Mozilla/5.0",
                "Referer": "https://www.moma.org/"
                }
    
    response = requests.get(image_url, headers=headers)
            
    img = Image.open(BytesIO(response.content))
    img.thumbnail((500, 500))
    if i<1000 :
        img.save(f"./Donnees/GAN_train/{row["ObjectID"]}.png")
    else : 
        img.save(f"./Donnees/GAN_test/{row["ObjectID"]}.png")


Title                   Simulated Dwelling for a Family of Five Project
Artist                                                      David Jacob
ConstituentID                                                      2864
ArtistBio                                         (American, born 1928)
Nationality                                                  (American)
BeginDate                                                        (1928)
EndDate                                                             (0)
Gender                                                           (male)
Date                                                               1970
Medium                Fiberglass, polyester resin, metal, paper, pla...
Dimensions                28 x 34 3/8 x 34 3/8" (71.1 x 87.3 x 87.3 cm)
CreditLine                          Estée and Joseph Lauder Design Fund
AccessionNumber                                                194.1973
Classification                                             Archi

In [27]:
!pip install torchvision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 54.3 MB/s  0:00:00


In [35]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

def get_dataloader(path, batch_size=32, image_size=64):

    transform = transforms.Compose([
        transforms.Resize(image_size),
        transforms.CenterCrop(image_size),
        transforms.RandomHorizontalFlip(0.5),
        transforms.ToTensor(),
        transforms.Normalize([0.5]*3, [0.5]*3)
    ])

    dataset = datasets.ImageFolder(root=path, transform=transform)

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=2,
        pin_memory=True
    )

    return loader

In [37]:
# models/generator.py
import torch.nn as nn

class Generator(nn.Module):
    def __init__(self, z_dim=100, channels=3, features=64):
        super().__init__()

        self.net = nn.Sequential(
            self.block(z_dim, features * 8, 4, 1, 0),   # 4x4
            self.block(features * 8, features * 4, 4, 2, 1),  # 8x8
            self.block(features * 4, features * 2, 4, 2, 1),  # 16x16
            self.block(features * 2, features, 4, 2, 1),      # 32x32
            nn.ConvTranspose2d(features, channels, 4, 2, 1),  # 64x64
            nn.Tanh()
        )

    def block(self, in_c, out_c, k, s, p):
        return nn.Sequential(
            nn.ConvTranspose2d(in_c, out_c, k, s, p, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(True)
        )

    def forward(self, x):
        return self.net(x)

In [38]:
# models/discriminator.py
import torch.nn as nn

class Discriminator(nn.Module):
    def __init__(self, channels=3, features=64):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv2d(channels, features, 4, 2, 1),
            nn.LeakyReLU(0.2),

            self.block(features, features * 2, 4, 2, 1),
            self.block(features * 2, features * 4, 4, 2, 1),
            self.block(features * 4, features * 8, 4, 2, 1),

            nn.Conv2d(features * 8, 1, 4, 1, 0),
            nn.Sigmoid()
        )

    def block(self, in_c, out_c, k, s, p):
        return nn.Sequential(
            nn.Conv2d(in_c, out_c, k, s, p, bias=False),
            nn.BatchNorm2d(out_c),
            nn.LeakyReLU(0.2)
        )

    def forward(self, x):
        return self.net(x).view(-1)

In [39]:
ls Donnees/GAN_train

100188.png  188788.png  34157.png   62014.png   79181.png   81202.png
1007.png    190112.png  34183.png   62040.png   79183.png   81209.png
102385.png  190933.png  3429.png    62093.png   79187.png   81213.png
102580.png  1911.png    3431.png    62197.png   79211.png   81223.png
102758.png  192291.png  34479.png   62223.png   79250.png   81225.png
102962.png  192892.png  3462.png    62298.png   79251.png   81232.png
104289.png  193590.png  3477.png    62419.png   79253.png   81233.png
104293.png  193811.png  35040.png   62894.png   79260.png   81242.png
104957.png  193812.png  35054.png   63050.png   79267.png   81253.png
105050.png  193813.png  35066.png   63051.png   79269.png   81268.png
105070.png  193814.png  35262.png   63052.png   79277.png   81295.png
105379.png  193816.png  35270.png   63260.png   79285.png   81307.png
105380.png  193818.png  35281.png   63263.png   79289.png   81329.png
105386.png  193819.png  35286.png   64278.png   79300.png   81368.png
105387.png  193820.p

In [45]:
# train.py
import torch
import torch.nn as nn
import torch.optim as optim

device = "cuda" if torch.cuda.is_available() else "cpu"

# Hyperparams
lr = 2e-4
batch_size = 32
z_dim = 100
epochs = 100

loader = get_dataloader("./Donnees/GAN_train", batch_size)

gen = Generator(z_dim).to(device)
disc = Discriminator().to(device)

opt_gen = optim.Adam(gen.parameters(), lr=lr, betas=(0.5, 0.999))
opt_disc = optim.Adam(disc.parameters(), lr=lr, betas=(0.5, 0.999))

criterion = nn.BCEWithLogitsLoss()

for epoch in range(epochs):
    for real, _ in loader:
        real = real.to(device)
        cur_batch = real.size(0)

        noise = torch.randn(cur_batch, z_dim, 1, 1).to(device)
        fake = gen(noise)

        # --- Train Discriminator ---
        disc_real = disc(real)
        loss_real = criterion(disc_real, torch.ones_like(disc_real) * 0.9)

        disc_fake = disc(fake.detach())
        loss_fake = criterion(disc_fake, torch.zeros_like(disc_fake))

        loss_disc = (loss_real + loss_fake) / 2

        disc.zero_grad()
        loss_disc.backward()
        opt_disc.step()

        # Train Generator (2 fois)
        for _ in range(2):
            noise = torch.randn(cur_batch, z_dim, 1, 1).to(device)
            fake = gen(noise)
        
            output = disc(fake)
            loss_gen = criterion(output, torch.ones_like(output))
        
            gen.zero_grad()
            loss_gen.backward()
            opt_gen.step()


    print(f"Epoch {epoch} | D: {loss_disc:.4f} | G: {loss_gen:.4f}")

torch.save(gen.state_dict(), "generator.pth")

Epoch 0 | D: 0.8090 | G: 0.4761
Epoch 1 | D: 0.6904 | G: 0.6580
Epoch 2 | D: 0.6046 | G: 0.6372
Epoch 3 | D: 0.5990 | G: 0.6773
Epoch 4 | D: 0.5583 | G: 0.6732
Epoch 5 | D: 0.5574 | G: 0.6256
Epoch 6 | D: 0.5697 | G: 0.6696
Epoch 7 | D: 0.6027 | G: 0.6927
Epoch 8 | D: 0.5588 | G: 0.6875
Epoch 9 | D: 0.5574 | G: 0.6930
Epoch 10 | D: 0.5535 | G: 0.6929
Epoch 11 | D: 0.5534 | G: 0.6927
Epoch 12 | D: 0.5537 | G: 0.6930
Epoch 13 | D: 0.5534 | G: 0.6926
Epoch 14 | D: 0.5535 | G: 0.6931
Epoch 15 | D: 0.5534 | G: 0.6928
Epoch 16 | D: 0.5535 | G: 0.6930
Epoch 17 | D: 0.5536 | G: 0.6928
Epoch 18 | D: 0.5533 | G: 0.6928
Epoch 19 | D: 0.5534 | G: 0.6931
Epoch 20 | D: 0.5534 | G: 0.6930
Epoch 21 | D: 0.5535 | G: 0.6929
Epoch 22 | D: 0.5534 | G: 0.6930
Epoch 23 | D: 0.5533 | G: 0.6931
Epoch 24 | D: 0.5533 | G: 0.6926
Epoch 25 | D: 0.5534 | G: 0.6926
Epoch 26 | D: 0.5533 | G: 0.6930
Epoch 27 | D: 0.5535 | G: 0.6913
Epoch 28 | D: 0.5535 | G: 0.6929
Epoch 29 | D: 0.5544 | G: 0.6928
Epoch 30 | D: 0.5534

In [46]:
# generate.py
import torch
from torchvision.utils import save_image

device = "cuda" if torch.cuda.is_available() else "cpu"

gen = Generator().to(device)
gen.load_state_dict(torch.load("generator.pth", map_location=device))
gen.eval()

noise = torch.randn(16, 100, 1, 1).to(device)
fake = gen(noise)

save_image(fake, "samples.png", normalize=True)